# Module 4

## Prerequisites

In [1]:
import boto3
import json
import time
from datetime import datetime
import os

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

## Inference profile

In [2]:
prompt = """
What design patterns are used in this code?

You are an expert Python developer. Here's a codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

### Invoking cross-region inference profile

In [3]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "6d770c8f-93aa-4205-a736-bfa2d7871dd7",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 29 Sep 2026 13:44:19 GMT",
      "content-type": "application/json",
      "content-length": "5380",
      "connection": "keep-alive",
      "x-amzn-requestid": "6d770c8f-93aa-4205-a736-bfa2d7871dd7"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Given the provided code snippet and its context within a larger system with many similar classes, several design patterns may be applicable. Here's a breakdown of potential design patterns used in this `DataProcessor` class:\n\n### 1. **Factory Pattern**\nIf the system has multiple types of data processors or configurations, a Factory Pattern might be used to instantiate these objects. This pattern would help in managing the creation of `DataProcessor` instances based on different configurations or type

### Use your own inference profile

In [4]:
# create an inference profile
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

response = bedrock_control.create_inference_profile(
    inferenceProfileName='dk-nova-lite-profile',
    description='Custom inference profile for Nova Lite',
    modelSource={
        'copyFrom': 'arn:aws:bedrock:us-east-1:767397992380:inference-profile/us.amazon.nova-lite-v1:0'
    }
)
inference_profile_arn = response['inferenceProfileArn']

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "d2f1e4b2-756d-4759-853c-a6889457322a",
    "HTTPStatusCode": 201,
    "HTTPHeaders": {
      "date": "Tue, 29 Sep 2026 13:45:05 GMT",
      "content-type": "application/json",
      "content-length": "125",
      "connection": "keep-alive",
      "x-amzn-requestid": "d2f1e4b2-756d-4759-853c-a6889457322a"
    },
    "RetryAttempts": 0
  },
  "inferenceProfileArn": "arn:aws:bedrock:us-east-1:767397992380:application-inference-profile/659l66ybzyyq",
  "status": "ACTIVE"
}


In [5]:
response = bedrock.converse(
    modelId=inference_profile_arn,
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "70f87663-c3f3-4147-81f9-b75d733990ae",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 29 Sep 2026 13:46:05 GMT",
      "content-type": "application/json",
      "content-length": "4442",
      "connection": "keep-alive",
      "x-amzn-requestid": "70f87663-c3f3-4147-81f9-b75d733990ae"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Given the context and the structure of the `DataProcessor` class, let's identify the design patterns that may be in use or could be applied to improve the design.\n\n### Existing Patterns\n\n1. **Builder Pattern**:\n   - The `__init__` method initializes the `DataProcessor` with a `config`. This could potentially be a form of the Builder pattern, where the object is constructed using a configuration object.\n   - If the configuration object is complex and has multiple steps to create it, a Builder patte

In [8]:
response = bedrock_control.delete_inference_profile(
    inferenceProfileIdentifier=inference_profile_arn
)

## Batch Inference

Upload jsonl file to S3

In [7]:
bucket_name = "batch-data-inference-dk0929"

s3 = boto3.client('s3')

# Upload manifest to S3
input_key = 'input/city-reviews-manifest.jsonl'
s3.upload_file('input/city-reviews-manifest.jsonl', bucket_name, input_key)

Call the CreateModelInvocationJob API

In [8]:
accountnumber = "767397992380"

# Create the bedrock control plane client
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

# Create batch inference job
job_name = f"batch-job-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

response = bedrock_control.create_model_invocation_job(
    jobName=job_name,
    roleArn=f"arn:aws:iam::{accountnumber}:role/BedrockBatchRole",
    modelId="us.amazon.nova-lite-v1:0",
    inputDataConfig={
        's3InputDataConfig': {
            's3Uri': f's3://{bucket_name}/input/'
        }
    },
    outputDataConfig={
        's3OutputDataConfig': {
            's3Uri': f's3://{bucket_name}/output/'
        }
    }
)

job_arn = response['jobArn']


Get the Job Status

In [9]:
status_response = bedrock_control.get_model_invocation_job(jobIdentifier=job_arn)
status = status_response['status']
print(f"Job status: {status}")

Job status: Submitted


Instead of waiting, let's use a jobArn for a job we know already finished

In [10]:
completed_job_arn = job_arn

Wait for the job to finish

In [ ]:
# Wait for job completion
while True:
    status_response = bedrock_control.get_model_invocation_job(jobIdentifier=completed_job_arn)
    status = status_response['status']
    print(f"Job status: {status}")
    
    if status in ['Completed', 'PartiallyCompleted', 'Failed', 'Stopped', 'Expired']:
        break
    
    time.sleep(30)



Job status: Submitted
Job status: Submitted
Job status: Submitted
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Scheduled
Job status: Scheduled


Download the result

In [18]:
if status in ["Completed", "PartiallyCompleted"]:
    # Download results
    output_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix='output/')
    
    os.makedirs('batch-output', exist_ok=True)
    
    for obj in output_objects.get('Contents', []):
        if obj['Key'].endswith('/'):
            continue
            
        local_file = os.path.join('batch-output', obj['Key'].split('/')[-1])
        s3.download_file(bucket_name, obj['Key'], local_file)
        print(f"Downloaded: {local_file}")


Downloaded: batch-output\city-reviews-manifest.jsonl.out
Downloaded: batch-output\manifest.json.out


Stop the job we created to save on cost

In [ ]:
response = bedrock_control.stop_model_invocation_job(
    jobIdentifier=job_arn
)

## Prompt Caching with Nova Lite

In [2]:
# Large context that we want to cache
cached_context = """
You are a expert Python developer. Here's a large codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

First request that establishes the cache

In [ ]:
firstRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default", "ttl": "1h"
                    }
                },
                {
                    "text": "What design patterns are used in this code?"
                }
          ]
      }
    ]
)

print(json.dumps(firstRequest, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "9e7e4d28-d9aa-4260-8615-a2a5a23c7565",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 18 Sep 2026 16:35:07 GMT",
      "content-type": "application/json",
      "content-length": "6099",
      "connection": "keep-alive",
      "x-amzn-requestid": "9e7e4d28-d9aa-4260-8615-a2a5a23c7565"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet, we can identify several design patterns that are being used:\n\n1. **Factory Method Pattern**:\n    - Although not explicitly shown in the snippet, the `__init__` method that takes a `config` parameter suggests the use of a factory method. This pattern is often used to create objects without specifying the exact class of object that will be created. The `config` parameter could be used to initialize different types of `DataProcessor` subclasses with different configur

Second request that uses cached context

In [4]:
secondRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "what else can you tell me about this code?"
                }
          ]
      }
    ]
)

print(json.dumps(secondRequest, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "d5584a24-f24b-4098-ac96-d5061ac38a5d",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 18 Sep 2026 16:36:39 GMT",
      "content-type": "application/json",
      "content-length": "5648",
      "connection": "keep-alive",
      "x-amzn-requestid": "d5584a24-f24b-4098-ac96-d5061ac38a5d"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Given the context provided, here are several insights and recommendations that can be applied to the `DataProcessor` class and the larger system it is part of:\n\n### Class Design and Structure\n\n1. **Initialization and Configuration:**\n   - The `__init__` method initializes the `config` and an empty list `self.data`. This suggests that `config` is a crucial component for the class's operation. It's important to ensure that `config` is properly validated and documented.\n   - Consider adding type hint

Compare the usage

In [5]:
print("### First request ###")
print("Usage:", json.dumps(firstRequest['usage'], indent=2))
print("Metrics:", json.dumps(firstRequest['metrics'], indent=2))

print("\n\n### Second request ###")
print("Usage:", json.dumps(secondRequest['usage'], indent=2))
print("Metrics:", json.dumps(secondRequest['metrics'], indent=2))

### First request ###
Usage: {
  "inputTokens": 9,
  "outputTokens": 1252,
  "totalTokens": 1385,
  "cacheReadInputTokens": 0,
  "cacheWriteInputTokens": 124
}
Metrics: {
  "latencyMs": 7385
}


### Second request ###
Usage: {
  "inputTokens": 10,
  "outputTokens": 1101,
  "totalTokens": 1235,
  "cacheReadInputTokens": 124,
  "cacheWriteInputTokens": 0
}
Metrics: {
  "latencyMs": 5243
}


## Converse on documents
### Using `bytes` as a source

In [ ]:
with open('input/AnyCompany_financial_10K.pdf', "rb") as file:
    doc_bytes = file.read()

In [ ]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": "AnyCompany_financial",
                        "source": {
                            "bytes": doc_bytes
                        }
                    }
                },
                {
                    "text": "What investments have AnyCompany made?"
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

### Using s3Location as a source

In [ ]:
bucket_name = "batch-data-inference-dk01"

s3 = boto3.client('s3')

input_key = 'AnyCompany_financial_10K.pdf'
s3.upload_file('input/AnyCompany_financial_10K.pdf', bucket_name, input_key)

## Conversation history

### Create the DynamoDB Table and set the time to live

In [6]:
table_name = "conversation-history"

dynamodb = boto3.client('dynamodb', region_name="us-east-2")
response = dynamodb.create_table(
    TableName=table_name,
    BillingMode='PAY_PER_REQUEST',
    AttributeDefinitions=[
        {
            'AttributeName': 'userId',
            'AttributeType': 'S'
        },
        {
            'AttributeName': 'timestamp',
            'AttributeType': 'N'
        }
    ],
    KeySchema=[
        {
            'AttributeName': 'userId',
            'KeyType': 'HASH'
        },
        {
            'AttributeName': 'timestamp',
            'KeyType': 'RANGE'
        }
    ]
)

# Wait for table to be active
waiter = dynamodb.get_waiter('table_exists')
waiter.wait(TableName=table_name)

ttl_response = dynamodb.update_time_to_live(
    TableName=table_name,
    TimeToLiveSpecification={
        'AttributeName': 'ttl',
        'Enabled': True
    }
)

### Create the functions to get the conversation history and store new messages

In [7]:
table_name = "conversation-history"
ddbclient = boto3.client('dynamodb', region_name = "us-east-2")

def get_conversation_history(user_id):
    # Pagination is required in case the conversation is more than 1MB.
    paginator = ddbclient.get_paginator('query')
    pages = paginator.paginate(
        TableName = table_name,
        KeyConditionExpression = 'userId = :val',
        ExpressionAttributeValues = {':val': {'S': user_id}}
    )
    messages = []
    for page in pages:
        for item in page.get('Items', []):
            messages.append({
                "role": item["role"]["S"],
                "content": [{ "text": item["message"]["S"] }]
            })
    return messages

In [8]:
dynamodbResource = boto3.resource('dynamodb', region_name = "us-east-2")
conversation_table = dynamodbResource.Table(table_name)

def store_message(user_id, message, role):
    now_in_seconds = int(time.time())
    expire_ttl = now_in_seconds + (1 * 24 * 60 * 60) # 1 day
    
    conversation_table.put_item(
        Item = {
            'userId': user_id,
            'timestamp': now_in_seconds,
            'message': message,
            'role': role,
            'ttl': expire_ttl
        }
    )


### Create the function to send messages to the LLM

In [9]:
def send_message(user_id, message):
    
    # Load the conversation history
    conversation_messages = get_conversation_history(user_id)

    # Store the new message
    store_message(user_id, message, "user")
    
    # Add the new message to the conversation
    conversation_messages.append({
        "role": "user",
        "content": [{ "text": message }]
    })

    # Send the conversation to the LLM
    model_response = bedrock.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=conversation_messages,
        system = [{ 
            "text": "Please provide a helpful, conversational response based on the available information and conversation history." 
        }],
        inferenceConfig = {
            "maxTokens": 300,
            "temperature": 0.7,
            "topP": 0.9
        }
    )

    # Parse the message from the LLM, store it and return it
    assistant_msg = model_response['output']['message']['content'][0]['text']
    store_message(user_id, assistant_msg, "assistant")

    return conversation_messages, assistant_msg

### Sending messages

In [10]:
user_id = "Disha"
convo, response = send_message(user_id, "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?")
print(response)


Hello Disha! I'm glad to hear that you love pizza. Vancouver is a fantastic city for pizza lovers, with a variety of places to choose from. Here are some highly recommended pizza spots in the city:

### Downtown and Nearby Areas
1. **Libretto Ristorante**
   - **Location:** 119 W Pender St
   - **Specialty:** Wood-fired Neapolitan pizzas
   - **Atmosphere:** Elegant and sophisticated

2. **Cicin**
   - **Location:** 641 Davie St
   - **Specialty:** Classic wood-fired pizzas
   - **Atmosphere:** Casual and family-friendly

3. **Forno Wood Oven Bistro**
   - **Location:** 1438 Robson St
   - **Specialty:** Wood-fired pizzas and Italian dishes
   - **Atmosphere:** Cozy and inviting

### Kitsilano
1. **The Pizza Press**
   - **Location:** 5125 Dunbar St
   - **Specialty:** High-quality ingredients and creative toppings
   - **Atmosphere:** Casual and vibrant

2. **Cin Cin Pizza**
   - **Location:** 1026 W Broadway
   - **Specialty:** Wood-fired pizzas
   - **Atmosphere:** Friendly and laid

In [11]:
convo, response = send_message(user_id, "What do I love to eat?")
print(response)

You mentioned in your initial message that you love to eat pizza. If you have other favorite foods or would like recommendations for different types of cuisine, feel free to let me know! Whether you're in the mood for something sweet, savory, or anything in between, I'd be happy to help you find some delicious options.


In [12]:
print(json.dumps(convo, indent=2))

[
  {
    "role": "user",
    "content": [
      {
        "text": "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
        "text": "Hello Disha! I'm glad to hear that you love pizza. Vancouver is a fantastic city for pizza lovers, with a variety of places to choose from. Here are some highly recommended pizza spots in the city:\n\n### Downtown and Nearby Areas\n1. **Libretto Ristorante**\n   - **Location:** 119 W Pender St\n   - **Specialty:** Wood-fired Neapolitan pizzas\n   - **Atmosphere:** Elegant and sophisticated\n\n2. **Cicin**\n   - **Location:** 641 Davie St\n   - **Specialty:** Classic wood-fired pizzas\n   - **Atmosphere:** Casual and family-friendly\n\n3. **Forno Wood Oven Bistro**\n   - **Location:** 1438 Robson St\n   - **Specialty:** Wood-fired pizzas and Italian dishes\n   - **Atmosphere:** Cozy and inviting\n\n### Kitsilano\n1. **The Pizza Press**\